# Adam Optimizer

**难度:** Medium

从零实现 Adam 优化器。

Adam 结合了动量（一阶矩）和 RMSProp（二阶矩），并通过偏差校正实现自适应的逐参数学习率。

**签名:** `MyAdam(params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8)`

**方法:**
- `step()` — 使用偏差校正后的矩更新参数
- `zero_grad()` — 将所有参数梯度清零

**约束:**
- 必须与 `torch.optim.Adam` 数值一致
- 偏差校正: `m_hat = m / (1 - beta1^t)`

**提示:**
1. m = β1·m + (1-β1)·g,  v = β2·v + (1-β2)·g²
2. 偏差校正：m̂ = m/(1-β1ᵗ),  v̂ = v/(1-β2ᵗ)
3. p -= lr · m̂ / (√v̂ + ε)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

In [ ]:
# ✅ INTERVIEW

# ---- 手写 Adam：面试高频题 ----
# 核心公式（3 个）：
#   m_t = β1·m_{t-1} + (1-β1)·g_t          （一阶矩更新）
#   v_t = β2·v_{t-1} + (1-β2)·g_t²         （二阶矩更新）
#   p_t = p_{t-1} - α · m̂_t / (√v̂_t + ε)  （参数更新）
# 其中 m̂ = m/(1-β1^t), v̂ = v/(1-β2^t)      （偏差校正）

class MyAdam:
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8):
        # 面试要点：必须把 params 转为 list，否则生成器只能遍历一次
        self.params = list(params)
        self.lr = lr
        self.beta1, self.beta2 = betas
        self.eps = eps
        self.t = 0
        # 面试要点：用 zeros_like 保持与参数相同的 shape 和 device
        self.m = [torch.zeros_like(p) for p in self.params]
        self.v = [torch.zeros_like(p) for p in self.params]

    def step(self):
        self.t += 1
        # 面试要点：必须用 no_grad()，否则 inplace 操作会报错
        with torch.no_grad():
            for i, p in enumerate(self.params):
                if p.grad is None:
                    continue

                g = p.grad

                # 一阶矩：梯度的指数移动平均
                self.m[i] = self.beta1 * self.m[i] + (1 - self.beta1) * g
                # 二阶矩：梯度平方的指数移动平均
                self.v[i] = self.beta2 * self.v[i] + (1 - self.beta2) * g ** 2

                # 偏差校正：补偿初始零值导致的偏差
                # 数学推导：E[m_t] = (1-β1^t)·E[g]，所以 m/(1-β1^t) 是无偏估计
                m_hat = self.m[i] / (1 - self.beta1 ** self.t)
                v_hat = self.v[i] / (1 - self.beta2 ** self.t)

                # 参数更新：注意是 inplace 操作 p -= ...
                p -= self.lr * m_hat / (torch.sqrt(v_hat) + self.eps)

    def zero_grad(self):
        for p in self.params:
            if p.grad is not None:
                p.grad.zero_()

In [ ]:
# Demo
torch.manual_seed(0)
w = torch.randn(4, 3, requires_grad=True)
opt = MyAdam([w], lr=0.01)
for i in range(5):
    loss = (w ** 2).sum()
    loss.backward()
    opt.step()
    opt.zero_grad()
    print(f'Step {i}: loss={loss.item():.4f}')

In [ ]:
from torch_judge import check
check('adam')

## 📝 核心思路总结

1. **Adam = 动量 + RMSProp + 偏差校正**：一阶矩 m 跟踪梯度均值，二阶矩 v 跟踪梯度方差
2. **偏差校正是关键**：初始 m=v=0 导致前几步估计偏小，除以 (1-β^t) 修正
3. **逐参数自适应学习率**：梯度大的参数 v 大→更新步长小，反之亦然